# FedGen multi-ancestry sites: Kaggle test

Generates 3 federated sites, one per ancestry (site 1 EUR, site 2 EAS, site 3 AFR), then runs a per-site GWAS, a meta-analysis, and a Manhattan plot.

**Before running:** in the notebook settings, turn **Internet on** (needed to download FedGen and LDAK). A CPU session is enough.

All sample sizes, SNP counts and causal-SNP settings live in **one place**: the Parameters cell right below. Run the cells in order.

In [ ]:

import os

# One site per ancestry: site 1 EUR, site 2 EAS, site 3 AFR.
# This is the ONLY place sample/SNP counts are set -- change them here, nowhere else.
os.environ["SAMPLES"] = "100000 95000 110000"
os.environ["NSNPS"] = "500000 480000 520000"
os.environ["ANCESTRY"] = "EUR EAS AFR"

# For a quick test instead, comment the block above and uncomment this
# (full size needs ~38 GB disk and ~20 min/site; this test needs neither):
# os.environ["SAMPLES"] = "5000 5000 5000"
# os.environ["NSNPS"] = "50000 50000 50000"

# Causal SNPs shared by all ancestries, and unique to each ancestry
# (one number for every ancestry, or per ancestry, e.g. "EUR=5,EAS=5,AFR=10")
os.environ["SHARED_CAUSALS"] = "20"
os.environ["UNIQUE_CAUSALS"] = "0"

os.environ["RUN_GWAS"] = "1"
os.environ["PYTHON"] = "python"

for k in ("SAMPLES", "NSNPS", "ANCESTRY", "SHARED_CAUSALS", "UNIQUE_CAUSALS"):
    print(f"{k}={os.environ[k]}")

## 2. Download FedGen and LDAK, check the machine

In [ ]:
%%bash
cd /kaggle/working
[ -d FedGen ] || git clone -q https://github.com/collaborativebioinformatics/FedGen.git
cd FedGen
mkdir -p tools
# LDAK's GitHub now only has 6.3; save it under the name the script expects
[ -f tools/ldak6.1.linux ] || curl -sSL -o tools/ldak6.1.linux https://github.com/dougspeed/LDAK/raw/main/ldak6.3.linux
chmod +x tools/ldak6.1.linux
./tools/ldak6.1.linux 2>&1 | head -5
python -c "import numpy, sys; print('python', sys.version.split()[0], 'numpy', numpy.__version__)"
echo "cores: $(nproc)"; free -g | head -2; df -h /kaggle/working | tail -1

## 3. Write the multi-ancestry simulation tool

In [ ]:
%%writefile /kaggle/working/FedGen/tools/ancestry_sim.py
#!/usr/bin/env python3
"""Multi-ancestry genotypes for FedGen sites, to pair with LDAK --make-phenos.

  reference  shared SNPs, per-ancestry allele frequencies (Balding-Nichols tree), causal SNPs
             shared by all ancestries (effects correlated --rg) or unique to one ancestry
             -> causals_<ANC>.txt, effects_<ANC>.txt, causal_types.tsv
  site       one site's .bed/.bim/.fam/.covar with its ancestry's frequencies and LD
  gwas       covariate-adjusted linear GWAS of an LDAK .pheno (optional)
  meta       fixed-effect meta-analysis of site GWAS files with I^2
"""
import argparse
import math
import os
import shutil
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
from statistics import NormalDist

import numpy as np

# name: (parent, Fst from parent, LD autocorrelation, mean LD-block length in SNPs); parents first
TREE = {"AFR": ("ANC", 0.07, 0.80, 25), "OOA": ("ANC", 0.14, 0, 1),
        "EUR": ("OOA", 0.08, 0.92, 60), "EAS": ("OOA", 0.12, 0.93, 70)}
ANCESTRIES = ["AFR", "EUR", "EAS"]
MAGIC = bytes([0x6C, 0x1B, 0x01])
ENCODE = np.array([3, 2, 0], np.uint8)       # A1 count 0/1/2 -> PLINK 2-bit code
DECODE = np.array([2, 0, 1, 0], np.float64)  # PLINK code -> A1 count (missing not expected)
SHIFTS = np.array([0, 2, 4, 6], np.uint8)
norm_ppf = np.vectorize(NormalDist().inv_cdf, otypes=[float])
_erfc = np.frompyfunc(math.erfc, 1, 1)


def pvalue(z):
    return _erfc(np.abs(z) / math.sqrt(2)).astype(float)


def reference(a):
    rng = np.random.default_rng(a.seed)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    m = a.num_snps
    chrom = np.sort(rng.integers(1, 23, m))
    pos = np.concatenate([np.cumsum(rng.integers(1000, 20000, (chrom == c).sum())) for c in range(1, 23)])
    alleles = np.array(list("ACGT"))[np.argsort(rng.random((m, 4)), 1)[:, :2]]
    maf = rng.uniform(0.01, 0.5, m)
    freq = {"ANC": np.where(rng.random(m) < 0.5, maf, 1 - maf)}
    for name, (parent, fst, _, _) in TREE.items():
        p, k = np.clip(freq[parent], 1e-9, 1 - 1e-9), (1 - fst) / fst
        freq[name] = rng.beta(p * k, (1 - p) * k)

    # Shared causals: MAF >= 1% in every ancestry, effects correlated rg across ancestries.
    # Unique causals: MAF >= 1% in their own ancestry, causal (non-zero effect) only there.
    maf = {x: np.minimum(freq[x], 1 - freq[x]) for x in ANCESTRIES}
    pool = np.flatnonzero(np.min(list(maf.values()), 0) >= 0.01)
    causal = {"shared": rng.choice(pool, a.num_shared_causals, replace=False)}
    for x in ANCESTRIES:
        pool = np.setdiff1d(np.flatnonzero(maf[x] >= 0.01), np.concatenate(list(causal.values())))
        causal[x] = rng.choice(pool, a.num_unique_causals[x], replace=False)
    cov = np.full((3, 3), a.rg) + (1 - a.rg) * np.eye(3)
    z_shared = rng.multivariate_normal(np.zeros(3), cov, (a.num_phenos, a.num_shared_causals))

    names = np.array([f"rs{c}_{b}" for c, b in zip(chrom, pos)])
    for k, x in enumerate(ANCESTRIES):
        idx = np.concatenate([causal["shared"], causal[x]])
        z = np.concatenate([z_shared[:, :, k], rng.standard_normal((a.num_phenos, len(causal[x])))], 1)
        p = freq["ANC"][idx]
        np.savetxt(out / f"effects_{x}.txt", z * (2 * p * (1 - p)) ** (a.power / 2), fmt="%.6g")
        (out / f"causals_{x}.txt").write_text((" ".join(names[idx]) + "\n") * a.num_phenos)
    with open(out / "causal_types.tsv", "w") as fh:
        fh.write("SNP\tTYPE\n" + "".join(f"{names[i]}\t{t}\n" for t, idx in causal.items() for i in idx))
    np.savez(out / "reference.npz", chrom=chrom, pos=pos, names=names, alleles=alleles,
             causal=np.sort(np.concatenate(list(causal.values()))), **{x: freq[x] for x in ANCESTRIES})


def per_ancestry(text):
    """'5' -> 5 for every ancestry; 'EUR=5,AFR=10' -> those counts, 0 for the rest."""
    if "=" not in text:
        return dict.fromkeys(ANCESTRIES, int(text))
    counts = dict.fromkeys(ANCESTRIES, 0)
    for item in text.split(","):
        name, value = item.split("=")
        if name.strip() not in counts:
            raise argparse.ArgumentTypeError(f"unknown ancestry {name!r}; use {ANCESTRIES}")
        counts[name.strip()] = int(value)
    return counts


def _chromosome(job):
    path, p, rho, block, n, seed = job
    rng = np.random.default_rng(seed)
    thr = norm_ppf(np.clip(p, 1e-12, 1 - 1e-12)).astype(np.float32)  # P(z < thr) = p
    z, noise = np.empty(2 * n, np.float32), np.empty(2 * n, np.float32)
    with open(path, "wb") as fh:
        for s in range(0, len(p), 256):
            g = np.empty((min(256, len(p) - s), n), np.int8)
            for i in range(len(g)):
                if s + i == 0 or rng.random() < 1 / block:  # new LD block
                    rng.standard_normal(dtype=np.float32, out=z)
                else:                                     # AR(1) along the chromosome
                    rng.standard_normal(dtype=np.float32, out=noise)
                    noise *= np.float32(math.sqrt(1 - rho ** 2))
                    z *= np.float32(rho)
                    z += noise
                hap = z < thr[s + i]
                g[i] = hap[:n]
                g[i] += hap[n:]  # int8 add (bool + bool would be OR)
            codes = np.pad(ENCODE[g], ((0, 0), (0, -n % 4))).reshape(len(g), -1, 4)
            fh.write((codes << SHIFTS).sum(2, dtype=np.uint8).tobytes())


def site(a):
    ref = np.load(Path(a.reference) / "reference.npz")
    rng = np.random.default_rng(a.seed)
    causal = ref["causal"]
    others = np.setdiff1d(np.arange(len(ref["names"])), causal)
    keep = np.sort(np.concatenate([causal, rng.choice(others, a.num_snps - len(causal), replace=False)]))
    chrom, p = ref["chrom"][keep], ref[a.ancestry][keep]
    _, _, rho, block = TREE[a.ancestry]

    stem, n = Path(a.out), a.num_samples
    stem.parent.mkdir(parents=True, exist_ok=True)
    seeds = np.random.SeedSequence([a.seed, ANCESTRIES.index(a.ancestry)]).spawn(22)
    jobs = [(f"{stem}.part{c}", p[chrom == c], rho, block, n, seeds[c - 1]) for c in range(1, 23)]
    with ProcessPoolExecutor(a.threads) as ex:
        list(ex.map(_chromosome, jobs))
    with open(f"{stem}.bed", "wb") as bed:
        bed.write(MAGIC)
        for job in jobs:
            with open(job[0], "rb") as part:
                shutil.copyfileobj(part, bed, 16 << 20)
            os.remove(job[0])

    pos = ref["pos"][keep]
    np.savetxt(f"{stem}.bim", np.column_stack([chrom, ref["names"][keep], pos / 1e6, pos, ref["alleles"][keep]]),
               fmt="%s", delimiter="\t")
    ids = np.char.add(f"{stem.name}_{a.ancestry}_", np.arange(1, n + 1).astype(str))
    sex, age = rng.integers(1, 3, n), rng.normal(55, 8, n).clip(40, 70).round(1)
    np.savetxt(f"{stem}.fam", np.column_stack([ids, ids, [0] * n, [0] * n, sex, [-9] * n]), fmt="%s")
    np.savetxt(f"{stem}.covar", np.column_stack([ids, ids, sex, age]), fmt="%s", header="FID IID sex age", comments="")


def read_table(path):
    """LDAK/PLINK phenotype or covariate file -> {IID: values} (NA -> nan)."""
    rows = [line.split() for line in open(path) if line.strip()]
    rows = rows[1:] if rows[0][0] in ("FID", "ID1") else rows
    return {r[1]: [np.nan if v == "NA" else float(v) for v in r[2:]] for r in rows}


def gwas(a):
    ids = np.array([line.split()[1] for line in open(f"{a.bfile}.fam")])
    bim = np.loadtxt(f"{a.bfile}.bim", dtype=str, ndmin=2)
    n, m = len(ids), len(bim)
    pheno, covar = read_table(a.pheno), read_table(a.covar)
    y = np.array([pheno.get(i, [np.nan])[0] for i in ids])
    keep = np.isfinite(y)
    C = np.column_stack([np.ones(keep.sum()), [covar[i] for i in ids[keep]]])
    Q = np.linalg.qr(C)[0]
    y = y[keep] - Q @ (Q.T @ y[keep])
    dof = len(y) - C.shape[1] - 1

    bed = np.memmap(f"{a.bfile}.bed", np.uint8, "r", 3, (m, (n + 3) // 4))
    beta, se, freq = np.empty(m), np.empty(m), np.empty(m)
    for s in range(0, m, 256):
        codes = (np.asarray(bed[s:s + 256])[..., None] >> SHIFTS) & 3
        x = DECODE[codes.reshape(len(codes), -1)[:, :n][:, keep]].T  # (samples, snps)
        freq[s:s + 256] = x.mean(0) / 2
        x -= x.mean(0)          # invariant SNPs become exactly 0 -> NaN results
        x -= Q @ (Q.T @ x)
        xx = (x * x).sum(0)
        with np.errstate(divide="ignore", invalid="ignore"):
            b = (x.T @ y) / xx
            beta[s:s + 256], se[s:s + 256] = b, np.sqrt((y @ y - b * b * xx) / dof / xx)

    causal = set(Path(a.causals).read_text().split()) if a.causals else set()
    is_causal = np.isin(bim[:, 1], list(causal)).astype(int)
    cols = [bim[:, 1], bim[:, 0], bim[:, 3], bim[:, 4], bim[:, 5], freq.round(5), [keep.sum()] * m,
            beta, se, pvalue(beta / se), is_causal]
    np.savetxt(a.out, np.column_stack(cols), fmt="%s", delimiter="\t", comments="",
               header="SNP\tCHR\tBP\tA1\tA2\tA1_FREQ\tN\tBETA\tSE\tP\tCAUSAL")


def summarise(name, z, causal, types):
    ok = np.isfinite(z)
    hit = ok & (pvalue(np.nan_to_num(z)) < 5e-8)
    lam = np.median(z[ok & ~causal] ** 2) / 0.4549
    found = "".join(f"{f'{(hit & m).sum()}/{m.sum()}':>9}" for m in types.values())
    print(f"{name:<12}{hit.sum():>9}{(hit & causal).sum():>8}{lam:>10.3f}{found}")


def meta(a):
    tabs = [np.genfromtxt(f, names=True, dtype=None, encoding=None) for f in a.gwas]
    snps = np.unique(np.concatenate([t["SNP"] for t in tabs]))
    B, S = np.full((2, len(tabs), len(snps)), np.nan)
    causal = np.zeros(len(snps), bool)
    for k, t in enumerate(tabs):
        i = np.searchsorted(snps, t["SNP"])
        ok = np.minimum(t["A1_FREQ"], 1 - t["A1_FREQ"]) >= 0.01
        B[k, i[ok]], S[k, i[ok]] = t["BETA"][ok], t["SE"][ok]
        causal[i] |= t["CAUSAL"] == 1

    types = {}  # causal SNPs found at p < 5e-8, per causal type (shared / unique to an ancestry)
    if a.truth:
        snp, kind = np.array([line.split() for line in open(a.truth)][1:]).T
        causal |= np.isin(snps, snp)
        types = {t: np.isin(snps, snp[kind == t]) for t in dict.fromkeys(kind)}
    print(f"{'study':<12}{'GWS hits':>9}{'causal':>8}{'lambdaGC':>10}" + "".join(f"{t:>9}" for t in types))
    for k, f in enumerate(a.gwas):
        summarise(Path(f).name.split(".")[0], B[k] / S[k], causal, types)

    with np.errstate(divide="ignore", invalid="ignore"):
        W, b = np.nan_to_num(1 / S ** 2), np.nan_to_num(B)
        beta, se = (W * b).sum(0) / W.sum(0), 1 / np.sqrt(W.sum(0))
        q, df = (W * (b - beta) ** 2).sum(0), (W > 0).sum(0) - 1
        i2 = np.nan_to_num(np.clip((q - df) / q, 0, 1))
    p = pvalue(beta / se)
    summarise("META", beta / se, causal, types)
    print(f"mean I^2: causal SNPs {i2[causal & (df > 0)].mean():.2f}, other SNPs {i2[~causal & (df > 0)].mean():.2f}")
    np.savetxt(a.out, np.column_stack([snps, df + 1, beta, se, p, q, i2, causal.astype(int)]), fmt="%s",
               delimiter="\t", comments="", header="SNP\tK\tBETA\tSE\tP\tQ\tI2\tCAUSAL")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    r = sub.add_parser("reference")
    r.add_argument("--out", required=True)
    r.add_argument("--num-snps", type=int, default=520000)
    r.add_argument("--num-shared-causals", type=int, default=20, help="causal in every ancestry")
    r.add_argument("--num-unique-causals", type=per_ancestry, default="0",
                   help="causal in one ancestry only: one number for each ancestry, or e.g. EUR=5,EAS=5,AFR=10")
    r.add_argument("--num-phenos", type=int, default=1)
    r.add_argument("--power", type=float, default=-0.25)
    r.add_argument("--rg", type=float, default=0.8, help="cross-ancestry effect correlation")
    r.add_argument("--seed", type=int, default=2024)
    s = sub.add_parser("site")
    s.add_argument("--reference", required=True)
    s.add_argument("--ancestry", required=True, choices=ANCESTRIES)
    s.add_argument("--num-samples", type=int, required=True)
    s.add_argument("--num-snps", type=int, required=True)
    s.add_argument("--seed", type=int, required=True)
    s.add_argument("--threads", type=int, default=os.cpu_count())
    s.add_argument("--out", required=True)
    g = sub.add_parser("gwas")
    for opt in ("--bfile", "--pheno", "--covar", "--out"):
        g.add_argument(opt, required=True)
    g.add_argument("--causals", help="reference causals_<ANC>.txt (adds the CAUSAL column)")
    t = sub.add_parser("meta")
    t.add_argument("--gwas", nargs="+", required=True)
    t.add_argument("--out", required=True)
    t.add_argument("--truth", help="reference causal_types.tsv: count found causal SNPs by type")
    a = ap.parse_args()
    {"reference": reference, "site": site, "gwas": gwas, "meta": meta}[a.cmd](a)


if __name__ == "__main__":
    main()

## 4. Write the site generator script

In [ ]:
%%writefile /kaggle/working/FedGen/scripts/generate_federated_sites.sh
#!/bin/bash
# Generate a single federated learning site with genomic data for Parkinson's disease
# Usage: ./scripts/generate_federated_sites.sh <site_number>
# Example: ./scripts/generate_federated_sites.sh 1

# One site per ancestry: site 1 EUR, site 2 EAS, site 3 AFR
# Override via environment, e.g. SAMPLES="5000 5000 5000" NSNPS="50000 50000 50000"
# This is the ONLY place sample/SNP counts are defined; don't sed-edit this file.
read -ra samples <<< "${SAMPLES:-100000 95000 110000}"
read -ra nsnps <<< "${NSNPS:-500000 480000 520000}"
read -ra ancestry <<< "${ANCESTRY:-EUR EAS AFR}"
nsites=${#samples[@]}
if [ "${#nsnps[@]}" -ne "$nsites" ] || [ "${#ancestry[@]}" -ne "$nsites" ]; then
  echo "Error: SAMPLES (${#samples[@]}), NSNPS (${#nsnps[@]}), ANCESTRY (${#ancestry[@]}) must all have the same count"
  exit 1
fi
# Causal SNPs shared by all ancestries, and unique to each ancestry
# (one number for every ancestry, or per ancestry, e.g. UNIQUE_CAUSALS=EUR=5,EAS=5,AFR=10)
SHARED_CAUSALS="${SHARED_CAUSALS:-20}"
UNIQUE_CAUSALS="${UNIQUE_CAUSALS:-0}"

# Check if site number is provided
if [ $# -eq 0 ]; then
  echo "Error: Site number required"
  echo "Usage: ./scripts/generate_federated_sites.sh <site_number>"
  echo "Example: ./scripts/generate_federated_sites.sh 1"
  echo "Site number must be between 1 and ${nsites}"
  exit 1
fi

site=$1

# Validate site number
if ! [[ "$site" =~ ^[0-9]+$ ]] || [ "$site" -lt 1 ] || [ "$site" -gt ${nsites} ]; then
  echo "Error: Site number must be between 1 and ${nsites}"
  exit 1
fi

# LDAK binary path
case "$(uname -s)" in
    Darwin)
        export OS_TYPE="mac"
        ;;
    Linux*)
        export OS_TYPE="linux"
        ;;
    *)
        echo "Unsupported OS. Please use macOS or Linux."
        exit 1
        ;;
esac

LDAK="./tools/ldak6.1.${OS_TYPE}"
SIM="${PYTHON:-python3} ./tools/ancestry_sim.py"
REF_DIR="./data/simulated_sites/reference"

# Create output directory
OUTPUT_DIR="./data/simulated_sites/site${site}"
mkdir -p ${OUTPUT_DIR}

idx=$((site-1))
anc=${ancestry[$idx]}

echo "========================================"
echo "Federated Genomic Data Simulation"
echo "========================================"
echo ""
echo "Generating site ${site}: ${samples[$idx]} samples, ${nsnps[$idx]} SNPs, ancestry ${anc}"
echo "========================================"

# Shared SNPs, causal SNPs and per-ancestry effects (built once for all sites)
causal_settings="shared=${SHARED_CAUSALS} unique=${UNIQUE_CAUSALS}"
if [ ! -f ${REF_DIR}/reference.npz ]; then
  echo "Step 0: Building shared multi-ancestry reference (${causal_settings})..."
  $SIM reference --out ${REF_DIR} --num-shared-causals ${SHARED_CAUSALS} \
    --num-unique-causals ${UNIQUE_CAUSALS} --power -0.25 --num-phenos 1 || exit 1
  echo "${causal_settings}" > ${REF_DIR}/settings.txt
elif [ "$(cat ${REF_DIR}/settings.txt 2>/dev/null)" != "${causal_settings}" ]; then
  echo "Error: ${REF_DIR} was built with different causal settings; delete it to rebuild"
  exit 1
fi
num_causals=$(head -1 ${REF_DIR}/causals_${anc}.txt | wc -w | tr -d ' ')

# Generate ancestry-specific genotypes (also creates .covar file)
echo "Step 1: Generating ${anc} genotypes..."
$SIM site \
  --reference ${REF_DIR} \
  --ancestry ${anc} \
  --out ${OUTPUT_DIR}/site${site}_geno \
  --num-samples ${samples[$idx]} \
  --num-snps ${nsnps[$idx]} \
  --seed ${site}

# Generate Parkinson's disease phenotypes
echo "Step 2: Generating Parkinson's phenotypes..."
$LDAK \
  --make-phenos ${OUTPUT_DIR}/site${site}_pheno \
  --bfile ${OUTPUT_DIR}/site${site}_geno \
  --her 0.25 \
  --prevalence 0.01 \
  --num-causals ${num_causals} \
  --causals ${REF_DIR}/causals_${anc}.txt \
  --effects ${REF_DIR}/effects_${anc}.txt \
  --power -0.25 \
  --num-phenos 1 \
  --covar ${OUTPUT_DIR}/site${site}_geno.covar \
  --covar-her 0.1

# Optional per-site GWAS for meta-analysis
if [ "${RUN_GWAS:-0}" = "1" ]; then
  echo "Step 3: Per-site GWAS..."
  $SIM gwas --bfile ${OUTPUT_DIR}/site${site}_geno --pheno ${OUTPUT_DIR}/site${site}_pheno.pheno \
    --covar ${OUTPUT_DIR}/site${site}_geno.covar --causals ${REF_DIR}/causals_${anc}.txt --out ${OUTPUT_DIR}/site${site}.gwas.tsv
fi

echo ""
echo "========================================"
echo "Site ${site} generated successfully!"
echo "========================================"
echo ""
echo "Generated files:"
echo "  - Genotypes: ${OUTPUT_DIR}/site${site}_geno.bed/bim/fam"
echo "  - Phenotypes: ${OUTPUT_DIR}/site${site}_pheno.pheno"
echo "  - Covariates: ${OUTPUT_DIR}/site${site}_geno.covar"

## 5. Make the script executable

In [ ]:
%%bash
cd /kaggle/working/FedGen
chmod +x scripts/generate_federated_sites.sh

## 6. Generate the 3 sites

Uses the settings from the Parameters cell above. To change `SHARED_CAUSALS`/`UNIQUE_CAUSALS` after a run, delete `data/simulated_sites/reference` first.

In [ ]:
%%bash
cd /kaggle/working/FedGen
for s in 1 2 3; do
  time ./scripts/generate_federated_sites.sh $s || break
  if [ ! -f data/simulated_sites/site$s/site${s}_pheno.pheno ]; then
    echo "STOP: LDAK wrote no phenotype for site $s (see its messages above)"; break
  fi
done
ls -lh data/simulated_sites/*

## 7. Meta-analysis across the 3 sites

In [ ]:
%%bash
cd /kaggle/working/FedGen
python tools/ancestry_sim.py meta \
  --gwas data/simulated_sites/site*/site*.gwas.tsv \
  --truth data/simulated_sites/reference/causal_types.tsv \
  --out data/simulated_sites/meta.tsv

## 8. Manhattan plot

Visual check that the pipeline worked: causal SNPs (red) should cluster above the genome-wide significance line more than random chance, especially at full sample size. `meta.tsv` has no CHR/BP columns, so they're pulled from site1's GWAS file and merged in by SNP name.

In [ ]:

import csv
import numpy as np
import matplotlib.pyplot as plt

base = "/kaggle/working/FedGen/data/simulated_sites"

# meta.tsv has no CHR/BP; get them from any site's per-SNP GWAS output
pos = {}
with open(f"{base}/site1/site1.gwas.tsv") as fh:
    for row in csv.DictReader(fh, delimiter="\t"):
        pos[row["SNP"]] = (int(row["CHR"]), int(row["BP"]))

rows = []
with open(f"{base}/meta.tsv") as fh:
    for row in csv.DictReader(fh, delimiter="\t"):
        if row["SNP"] in pos:
            chrom, bp = pos[row["SNP"]]
            rows.append((chrom, bp, float(row["P"]), row["CAUSAL"] == "1"))
rows.sort(key=lambda r: (r[0], r[1]))

chrom = np.array([r[0] for r in rows])
bp = np.array([r[1] for r in rows])
p = np.array([r[2] for r in rows])
causal = np.array([r[3] for r in rows])

# cumulative x-position so chromosomes lay out left to right
offset, xpos, ticks = 0, np.empty(len(bp)), []
for ch in range(1, 23):
    m = chrom == ch
    if m.any():
        xpos[m] = bp[m] + offset
        ticks.append((offset + bp[m].mean(), str(ch)))
        offset = xpos[m].max() + 1_000_000

logp = -np.log10(np.clip(p, 1e-300, 1))
colors = np.where(chrom % 2 == 0, "#4C72B0", "#8C96C6")

plt.figure(figsize=(14, 4))
plt.scatter(xpos[~causal], logp[~causal], c=colors[~causal], s=6, alpha=0.6)
plt.scatter(xpos[causal], logp[causal], c="red", s=20, label="causal SNP", zorder=3)
plt.axhline(-np.log10(5e-8), color="grey", linestyle="--", linewidth=1, label="genome-wide significance")
plt.xticks(*zip(*ticks), rotation=90, fontsize=7)
plt.xlabel("Chromosome")
plt.ylabel("-log10(P)")
plt.title(f"Meta-analysis Manhattan plot ({causal.sum()} of {len(causal)} causal SNPs shown)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{base}/manhattan_meta.png", dpi=150)
plt.show()
print(f"Saved to {base}/manhattan_meta.png")

## 9. Check how LDAK handled the effect sizes


Open question: does LDAK use the effects in `effects_<ANC>.txt` as given, or rescale them (to match `--her 0.25`, or by allele frequency for `--power -0.25`)?
Compare the effect column of LDAK's `.effects` file with the values the tool wrote. A constant ratio means LDAK rescaled them.
Cases should be about 1% of samples.

In [ ]:
%%bash
cd /kaggle/working/FedGen/data/simulated_sites
echo "== LDAK effects, site 1 (EUR) =="; head -5 site1/site1_pheno.effects
echo "== effects given to LDAK (EUR) =="; tr ' ' '\n' < reference/effects_EUR.txt | head -5
echo "== causal SNPs given (EUR) =="; tr ' ' '\n' < reference/causals_EUR.txt | head -5
for s in 1 2 3; do awk -v s=$s '$3==1{c++} $3==0{n++} END{print "site" s ": " c+0 " cases, " n+0 " controls"}' site$s/site${s}_pheno.pheno; done

## 10. Does LDAK rescale effects uniformly, or by frequency too?

One matched SNP already showed LDAK shrinks our supplied effects (to hit `--her`). This checks whether that shrinkage is a single constant per site (relative effect sizes preserved) or varies with allele frequency (would mean `--power` is applied a second time on top of the `(2pq)^(power/2)` already baked into `effects_<ANC>.txt`).

In [ ]:

import csv
import numpy as np

base = "/kaggle/working/FedGen/data/simulated_sites"
anc = "EUR"

causal_names = open(f"{base}/reference/causals_{anc}.txt").read().split()
input_effects = dict(zip(causal_names, np.loadtxt(f"{base}/reference/effects_{anc}.txt", ndmin=1)))

ldak_effects, freq = {}, {}
with open(f"{base}/site1/site1_pheno.effects") as fh:
    header = fh.readline().split()
    for line in fh:
        row = dict(zip(header, line.split()))
        if row["Predictor"] in input_effects:
            ldak_effects[row["Predictor"]] = float(row["Effect"])
            freq[row["Predictor"]] = float(row["Centre"]) / 2  # Centre ~= 2*freq(A1)

snps = sorted(ldak_effects)
ratio = np.array([ldak_effects[s] / input_effects[s] for s in snps])
maf = np.array([min(freq[s], 1 - freq[s]) for s in snps])

print(f"{len(snps)} of {len(causal_names)} shared causal SNPs matched")
print(f"ratio mean={ratio.mean():.4f}  sd={ratio.std():.4f}  (sd/mean={ratio.std()/abs(ratio.mean()):.3f})")
print(f"correlation(ratio, MAF) = {np.corrcoef(ratio, maf)[0,1]:.3f}")
for s, r, m in zip(snps, ratio, maf):
    print(f"  {s:<20} ratio={r:+.4f}  MAF={m:.3f}")